# Phase 3 — Cross-linguistic topology feature comparison

Compares per-(lang × color) topology feature tensors produced by the mBERT attention
pipeline (mwk.2 thresholds, mwk.3 ripser+templates) across English, Spanish, and Russian.

**Scope:** COLOR domain only — COSI 115a May 2026 analysis.  
Emotion and kinship are out of scope; see `bd show ph-project-mwk` for rationale.

**Replaces:** the binary logistic regression in `replication/notebooks/features_prediction.ipynb`.  
The four pre-registered prediction tests (P1–P4) move to Phase 4 (`ph-project-bt5`).

**Inputs:**
- `results/phase3_thresholds/{lang}_color_*_bert-base-multilingual-cased.npy` — shape (12, 12, 6, N_kwic, 6)
- `results/phase3_ripser/{lang}_color_*_ripser.npy` — shape (12, 12, N_kwic, 14) *(if available)*
- `results/phase3_templates/{lang}_color_*_template.npy` — shape (12, 12, 6, N_kwic) *(if available)*

**Outputs:**
- `results/phase3_comparison/summary.csv` — per-(feature_type × pair) statistical summary
- `results/figures/phase3_thresholds_comparison_color.{pdf,png}` — comparison heatmap
- `results/figures/phase3_ripser_comparison_color.{pdf,png}` — *(if ripser available)*
- `results/figures/phase3_template_comparison_color.{pdf,png}` — *(if template available)*

## Setup

In [1]:
# Ensure the repo root is on sys.path (mirrors baseline A/B/C pattern)
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')

Repo root: ~/ph-project


In [2]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Constants ──────────────────────────────────────────────────────────────
REPO_ROOT     = _REPO_ROOT
RESULTS_DIR   = REPO_ROOT / 'results'
FIGURES_DIR   = RESULTS_DIR / 'figures'
COMP_DIR      = RESULTS_DIR / 'phase3_comparison'
THOLD_DIR     = RESULTS_DIR / 'phase3_thresholds'
RIPSER_DIR    = RESULTS_DIR / 'phase3_ripser'
TEMPLATE_DIR  = RESULTS_DIR / 'phase3_templates'

# ── Scope ──────────────────────────────────────────────────────────────────
# COSI 115a May 2026 rescoping: color domain only.
# See CLAUDE.md "Current scope" and `bd show ph-project-mwk`.
LANGUAGES     = ['en', 'es', 'ru']
DOMAINS       = ['color']      # one-line re-enable: ['color', 'emotion', 'kinship']

LANG_LABELS   = {'en': 'English', 'es': 'Spanish', 'ru': 'Russian'}
DOMAIN_LABELS = {'color': 'Color', 'emotion': 'Emotion', 'kinship': 'Kinship'}
PAIRS         = [('en', 'es'), ('en', 'ru'), ('ru', 'es')]
PAIR_LABELS   = {('en', 'es'): 'EN–ES', ('en', 'ru'): 'EN–RU', ('ru', 'es'): 'RU–ES'}
PAIR_COLORS   = {('en', 'es'): '#f28e2b', ('en', 'ru'): '#4e79a7', ('ru', 'es'): '#e15759'}

# ── Model / file-stem constants (must match mbert_attention_thresholds.ipynb) ──
MODEL_STEM    = 'bert-base-multilingual-cased'
MAX_LEN       = 32
N_LAYERS      = 12
N_HEADS       = 12

# Benjamini-Hochberg FDR threshold
FDR_Q         = 0.05

# Permutation test sample count
N_PERMUTATIONS = 1000

# ── Ensure output directories exist ────────────────────────────────────────
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
COMP_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.max_columns = 40
pd.options.display.width = 200

print('Setup complete.')
print(f'Figures will be written to: {FIGURES_DIR}')
print(f'Comparison summary will be written to: {COMP_DIR}')

Setup complete.
Figures will be written to: ~/ph-project/results/figures
Comparison summary will be written to: ~/ph-project/results/phase3_comparison


## Load feature tensors

Load per-(lang × color) topology feature tensors from disk.

- **Thresholds** (mwk.2): shape `(12, 12, 6, N_kwic, 6)` — layers × heads × thresholds × samples × stats
- **Ripser** (mwk.3): shape `(12, 12, N_kwic, 14)` — layers × heads × samples × ripser_features
- **Templates** (mwk.3): shape `(12, 12, 6, N_kwic)` — layers × heads × template_features × samples

Ripser and template tensors may not exist yet if `mbert_attention_ripser.ipynb` has not been
executed on the GPU. Missing feature types are skipped with a clear message rather than crashing.

In [3]:
def _find_npy(search_dir, lang, domain, suffix=''):
    """Return path to the .npy for (lang, domain) matching suffix, or None if absent.

    Matches files starting with '{lang}_{domain}_all_heads' and ending with suffix+'.npy'.
    """
    if not search_dir.is_dir():
        return None
    prefix = f'{lang}_{domain}_all_heads'
    for fname in sorted(search_dir.iterdir()):
        if fname.name.startswith(prefix) and fname.name.endswith(suffix + '.npy'):
            return fname
    return None


def _load_thresholds(lang, domain):
    """Load threshold feature tensor for (lang, domain).

    Shape: (12, 12, 6, N_kwic, 6) — layers × heads × thresholds × samples × stats.
    Returns None with a message if absent.
    """
    path = _find_npy(THOLD_DIR, lang, domain)
    if path is None:
        print(f'  [SKIP] phase3_thresholds/ output not found for ({lang!r}, {domain!r}) — '
              f're-run mbert_attention_thresholds.ipynb to produce it.')
        return None
    arr = np.load(path, allow_pickle=True)
    print(f'  ({lang}, {domain}) thresholds: {path.name}  shape={arr.shape}')
    return arr


def _load_ripser(lang, domain):
    """Load ripser feature tensor for (lang, domain).

    Shape: (12, 12, N_kwic, 14) — layers × heads × samples × ripser_features.
    Returns None with a message if absent.
    """
    path = _find_npy(RIPSER_DIR, lang, domain, suffix='_ripser')
    if path is None:
        print(f'  [SKIP] phase3_ripser/ output not found for ({lang!r}, {domain!r}) — '
              f're-run mbert_attention_ripser.ipynb to enable ripser comparison.')
        return None
    arr = np.load(path, allow_pickle=True)
    print(f'  ({lang}, {domain}) ripser:     {path.name}  shape={arr.shape}')
    return arr


def _load_template(lang, domain):
    """Load template feature tensor for (lang, domain).

    Shape: (12, 12, 6, N_kwic) — layers × heads × template_features × samples.
    Returns None with a message if absent.
    """
    path = _find_npy(TEMPLATE_DIR, lang, domain, suffix='_template')
    if path is None:
        print(f'  [SKIP] phase3_templates/ output not found for ({lang!r}, {domain!r}) — '
              f're-run mbert_attention_ripser.ipynb to enable template comparison.')
        return None
    arr = np.load(path, allow_pickle=True)
    print(f'  ({lang}, {domain}) template:   {path.name}  shape={arr.shape}')
    return arr


# ── Load all tensors ────────────────────────────────────────────────────────
# Keyed by (lang, domain) → np.ndarray or None
thresholds_tensors = {}
ripser_tensors     = {}
template_tensors   = {}

print('=== Loading feature tensors ===')
for domain in DOMAINS:
    for lang in LANGUAGES:
        print(f'\n[{lang}/{domain}]')
        thresholds_tensors[(lang, domain)] = _load_thresholds(lang, domain)
        ripser_tensors[(lang, domain)]     = _load_ripser(lang, domain)
        template_tensors[(lang, domain)]   = _load_template(lang, domain)

# Determine which feature types are available
_thold_available    = any(v is not None for v in thresholds_tensors.values())
_ripser_available   = any(v is not None for v in ripser_tensors.values())
_template_available = any(v is not None for v in template_tensors.values())

print()
print('Availability summary:')
print(f'  thresholds : {"YES" if _thold_available  else "NO — run mbert_attention_thresholds.ipynb first"}')
print(f'  ripser     : {"YES" if _ripser_available  else "NO — run mbert_attention_ripser.ipynb first"}')
print(f'  templates  : {"YES" if _template_available else "NO — run mbert_attention_ripser.ipynb first"}')

=== Loading feature tensors ===

[en/color]
  (en, color) thresholds: en_color_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_32_bert-base-multilingual-cased.npy  shape=(12, 12, 6, 2200, 6)
  [SKIP] phase3_ripser/ output not found for ('en', 'color') — re-run mbert_attention_ripser.ipynb to enable ripser comparison.
  [SKIP] phase3_templates/ output not found for ('en', 'color') — re-run mbert_attention_ripser.ipynb to enable template comparison.

[es/color]
  (es, color) thresholds: es_color_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_32_bert-base-multilingual-cased.npy  shape=(12, 12, 6, 2161, 6)
  [SKIP] phase3_ripser/ output not found for ('es', 'color') — re-run mbert_attention_ripser.ipynb to enable ripser comparison.
  [SKIP] phase3_templates/ output not found for ('es', 'color') — re-run mbert_attention_ripser.ipynb to enable template comparison.

[ru/color]
  (ru, color) thresholds: ru_color_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_L

## Per-(lang × domain) profile aggregation

For each (lang × domain) and each feature type, flatten the non-sample axes to produce
an `(N_kwic, F)` sample matrix, then compute a per-language mean profile (F-dimensional).

**Flattening strategy:**

| Feature type | Tensor shape | Flatten → (N_kwic, F) |
|---|---|---|
| thresholds | (12, 12, 6, N_kwic, 6) | F = 12×12×6×6 = 5184 |
| ripser | (12, 12, N_kwic, 14) | F = 12×12×14 = 2016 |
| templates | (12, 12, 6, N_kwic) | F = 12×12×6 = 864 |

For thresholds: axes (layer=0, head=1, threshold=2, stat=4) → move N_kwic (axis 3) to front,
then reshape remaining axes to a single F dimension.

In [4]:
def flatten_thresholds(arr):
    """Flatten threshold tensor to (N_kwic, F) as float64.

    Input shape: (12, 12, 6, N_kwic, 6) — layers × heads × thresholds × samples × stats
    Output shape: (N_kwic, 12*12*6*6) = (N_kwic, 5184)

    Cast to float64 before flatten: tensors are stored as float16 which overflows
    during L2-norm computation over 5184 dimensions (max val ~619, sum of squares
    ~2e8 per row — exceeds float16 range of ~65504).
    """
    # arr: (L, H, T, N, S) — cast immediately to avoid float16 overflow downstream
    arr_f64 = arr.astype(np.float64)
    arr_t = np.moveaxis(arr_f64, 3, 0)   # → (N, L, H, T, S)
    N = arr_t.shape[0]
    return arr_t.reshape(N, -1)           # → (N, L*H*T*S)


def flatten_ripser(arr):
    """Flatten ripser tensor to (N_kwic, F) as float64.

    Input shape: (12, 12, N_kwic, 14) — layers × heads × samples × ripser_features
    Output shape: (N_kwic, 12*12*14) = (N_kwic, 2016)
    """
    arr_f64 = arr.astype(np.float64)
    arr_t = np.moveaxis(arr_f64, 2, 0)   # → (N, L, H, R)
    N = arr_t.shape[0]
    return arr_t.reshape(N, -1)           # → (N, L*H*R)


def flatten_template(arr):
    """Flatten template tensor to (N_kwic, F) as float64.

    Input shape: (12, 12, 6, N_kwic) — layers × heads × template_features × samples
    Output shape: (N_kwic, 12*12*6) = (N_kwic, 864)
    """
    arr_f64 = arr.astype(np.float64)
    arr_t = np.moveaxis(arr_f64, 3, 0)   # → (N, L, H, T)
    N = arr_t.shape[0]
    return arr_t.reshape(N, -1)           # → (N, L*H*T)


# ── Per-(lang × domain) flattened sample matrices ──────────────────────────
# Keyed by (lang, domain) → (N_kwic, F) float64 array or None
flat_thresholds = {}
flat_ripser     = {}
flat_template   = {}

for domain in DOMAINS:
    for lang in LANGUAGES:
        t = thresholds_tensors.get((lang, domain))
        r = ripser_tensors.get((lang, domain))
        m = template_tensors.get((lang, domain))
        flat_thresholds[(lang, domain)] = flatten_thresholds(t) if t is not None else None
        flat_ripser[(lang, domain)]     = flatten_ripser(r)     if r is not None else None
        flat_template[(lang, domain)]   = flatten_template(m)   if m is not None else None

print('Flattened sample matrices (float64):')
for domain in DOMAINS:
    for lang in LANGUAGES:
        ft = flat_thresholds.get((lang, domain))
        fr = flat_ripser.get((lang, domain))
        fm = flat_template.get((lang, domain))
        shapes = {
            'thresholds': ft.shape if ft is not None else None,
            'ripser':     fr.shape if fr is not None else None,
            'template':   fm.shape if fm is not None else None,
        }
        print(f'  ({lang}, {domain}): {shapes}')

Flattened sample matrices (float64):
  (en, color): {'thresholds': (2200, 5184), 'ripser': None, 'template': None}
  (es, color): {'thresholds': (2161, 5184), 'ripser': None, 'template': None}
  (ru, color): {'thresholds': (2267, 5184), 'ripser': None, 'template': None}


## save_fig helper

Verbatim from baseline_comparison.ipynb — same export convention (PDF + PNG, results/figures/).

In [5]:
def save_fig(fig, stem: str) -> None:
    """Export a figure as both .pdf (vector) and .png (300 dpi) to FIGURES_DIR."""
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'  Saved {pdf_path.name}  +  {png_path.name}')


print('save_fig helper defined.')

save_fig helper defined.


## Statistical comparison: Mann-Whitney U + permutation tests

For each language pair (en-es, en-ru, ru-es) and each feature dimension:
1. **Mann-Whitney U** (two-sided, `scipy.stats.mannwhitneyu`) per feature.
2. **Benjamini-Hochberg FDR correction** across all features at q < 0.05.
3. **Permutation test** on the aggregate L2 profile distance: shuffle language
   labels, recompute mean-profile L2 distance, repeat n=1000 times.

Results are collected into a summary CSV per (feature_type × pair).

In [6]:
def _bh_correction(pvalues, alpha=FDR_Q):
    """Benjamini-Hochberg FDR correction.

    Returns a boolean mask of the same length: True where the feature passes
    the BH procedure at the given false discovery rate (alpha).

    Implementation follows the standard BH step-up procedure:
        1. Sort p-values ascending.
        2. For rank k (1-indexed), the BH threshold is k/m * alpha.
        3. Find the largest k where p_k <= k/m * alpha.
        4. Reject all hypotheses with rank <= that k.
    """
    pv = np.asarray(pvalues, dtype=float)
    m = len(pv)
    if m == 0:
        return np.array([], dtype=bool)
    order = np.argsort(pv)
    thresholds = (np.arange(1, m + 1) / m) * alpha
    # find the largest rank k where p_{(k)} <= threshold_k
    sig_at_rank = pv[order] <= thresholds
    # if any rank k is significant, all ranks <= k are also significant (monotone)
    # cumsum from the right to enforce the step-up property
    cummax_right = np.maximum.accumulate(sig_at_rank[::-1])[::-1]
    reject_order = cummax_right
    reject = np.zeros(m, dtype=bool)
    reject[order] = reject_order
    return reject


def _permutation_test_profile_dist(mat_a, mat_b, n_perm=N_PERMUTATIONS, seed=42):
    """Permutation test on the L2 distance between mean feature profiles.

    mat_a, mat_b: (N_a, F) and (N_b, F) sample matrices for two languages.

    Observed statistic: L2 distance between the per-language mean profiles.
    Null distribution: shuffle combined labels, recompute statistic n_perm times.

    Returns: observed_dist, p_value (fraction of permuted dists >= observed).
    """
    rng = np.random.default_rng(seed)
    mean_a = mat_a.mean(axis=0)
    mean_b = mat_b.mean(axis=0)
    observed = float(np.linalg.norm(mean_a - mean_b))

    # Combined pool for shuffling
    combined = np.concatenate([mat_a, mat_b], axis=0)
    n_a = len(mat_a)

    count_ge = 0
    for _ in range(n_perm):
        idx = rng.permutation(len(combined))
        perm_a = combined[idx[:n_a]]
        perm_b = combined[idx[n_a:]]
        perm_dist = float(np.linalg.norm(perm_a.mean(axis=0) - perm_b.mean(axis=0)))
        if perm_dist >= observed:
            count_ge += 1

    p_value = (count_ge + 1) / (n_perm + 1)   # +1 for continuity / conservatism
    return observed, p_value


def _mwu_per_feature(mat_a, mat_b):
    """Run Mann-Whitney U test for each feature column.

    Returns p-values array of length F.
    """
    F = mat_a.shape[1]
    pvalues = np.empty(F)
    for j in range(F):
        _, p = scipy_stats.mannwhitneyu(mat_a[:, j], mat_b[:, j], alternative='two-sided')
        pvalues[j] = p
    return pvalues


def _compare_pair(flat_lang, l1, l2, domain, feature_type):
    """Run the full comparison pipeline for one (lang_pair, domain, feature_type).

    flat_lang: dict mapping (lang, domain) → (N, F) array.
    Returns a dict with summary statistics, or None if tensors are missing.
    """
    mat_a = flat_lang.get((l1, domain))
    mat_b = flat_lang.get((l2, domain))
    if mat_a is None or mat_b is None:
        return None

    pair_str = f'{l1}-{l2}'

    # --- per-feature Mann-Whitney U ---
    pvalues  = _mwu_per_feature(mat_a, mat_b)
    reject   = _bh_correction(pvalues)
    n_sig    = int(reject.sum())

    # --- top-5 most-different features (by MWU p-value ascending) ---
    top5_idx = np.argsort(pvalues)[:5].tolist()

    # --- permutation test on aggregate profile distance ---
    obs_dist, perm_p = _permutation_test_profile_dist(mat_a, mat_b)

    # --- per-language mean profiles ---
    profile_a = mat_a.mean(axis=0)   # (F,)
    profile_b = mat_b.mean(axis=0)   # (F,)

    return {
        'feature_type':       feature_type,
        'pair':               pair_str,
        'domain':             domain,
        'lang_a':             l1,
        'lang_b':             l2,
        'n_features':         mat_a.shape[1],
        'n_kwic_a':           len(mat_a),
        'n_kwic_b':           len(mat_b),
        'n_sig_features_q05': n_sig,
        'frac_sig_features':  round(n_sig / mat_a.shape[1], 4),
        'perm_obs_dist':      round(obs_dist, 6),
        'perm_p_value':       round(perm_p, 4),
        'top5_feature_idx':   str(top5_idx),   # serialised as string for CSV
        '_profile_a':         profile_a,       # private — used for figure, not CSV
        '_profile_b':         profile_b,       # private
        '_pvalues':           pvalues,          # private
        '_reject':            reject,           # private
    }


print('Statistical comparison helpers defined.')
print(f'  FDR q threshold : {FDR_Q}')
print(f'  Permutation n   : {N_PERMUTATIONS}')

Statistical comparison helpers defined.
  FDR q threshold : 0.05
  Permutation n   : 1000


In [7]:
# ── Run comparisons for all available feature types ─────────────────────────
all_results = []   # list of result dicts (private keys stripped for CSV later)

FEAT_TYPE_MAP = [
    ('thresholds', flat_thresholds),
    ('ripser',     flat_ripser),
    ('templates',  flat_template),
]

for feat_type, flat_dict in FEAT_TYPE_MAP:
    # Check if this feature type has any available tensors
    available = any(v is not None for v in flat_dict.values())
    if not available:
        print(f'\n[{feat_type}] No tensors available — skipping all pairs.')
        continue

    print(f'\n=== {feat_type} ===')
    for domain in DOMAINS:
        for (l1, l2) in PAIRS:
            pair_str = f'{l1}-{l2}'
            result = _compare_pair(flat_dict, l1, l2, domain, feat_type)
            if result is None:
                print(f'  [{domain}] {pair_str}: missing tensor(s) — skipped.')
                continue
            all_results.append(result)
            print(f'  [{domain}] {pair_str}: '
                  f'n_sig={result["n_sig_features_q05"]}/{result["n_features"]} '
                  f'({result["frac_sig_features"]*100:.1f}%)  '
                  f'perm_p={result["perm_p_value"]:.4f}  '
                  f'obs_dist={result["perm_obs_dist"]:.4f}')

print(f'\nTotal results: {len(all_results)} (feature_type × pair × domain)')


=== thresholds ===


  [color] en-es: n_sig=3914/5184 (75.5%)  perm_p=0.0010  obs_dist=582.1389


  [color] en-ru: n_sig=4210/5184 (81.2%)  perm_p=0.0010  obs_dist=962.3330


  [color] ru-es: n_sig=4050/5184 (78.1%)  perm_p=0.0010  obs_dist=705.8880

[ripser] No tensors available — skipping all pairs.

[templates] No tensors available — skipping all pairs.

Total results: 3 (feature_type × pair × domain)


## Save summary CSV

Strip private keys (prefixed with `_`) and write the summary CSV to
`results/phase3_comparison/summary.csv`.

In [8]:
# ── Assemble summary CSV (strip private _ keys) ─────────────────────────────
PUBLIC_COLS = [
    'feature_type', 'pair', 'domain', 'lang_a', 'lang_b',
    'n_features', 'n_kwic_a', 'n_kwic_b',
    'n_sig_features_q05', 'frac_sig_features',
    'perm_obs_dist', 'perm_p_value',
    'top5_feature_idx',
]

if all_results:
    rows = [{k: v for k, v in r.items() if not k.startswith('_')} for r in all_results]
    df_summary = pd.DataFrame(rows)[PUBLIC_COLS]
    df_summary = df_summary.sort_values(['feature_type', 'domain', 'pair']).reset_index(drop=True)

    out_path = COMP_DIR / 'summary.csv'
    df_summary.to_csv(out_path, index=False)
    print(f'Written: {out_path}  ({len(df_summary)} rows)')
    df_summary
else:
    print('No results to save — all feature types were missing.')
    print('Run mbert_attention_thresholds.ipynb (and optionally mbert_attention_ripser.ipynb) first.')
    df_summary = pd.DataFrame(columns=PUBLIC_COLS)

Written: ~/ph-project/results/phase3_comparison/summary.csv  (3 rows)


## Comparison figures

One figure per available feature type: a heatmap of significant-feature counts and
permutation-test p-values across language pairs, analogous to `fig2_BC_*` in
`baseline_comparison.ipynb`.

Figure naming: `phase3_{feat_type}_comparison_{domain}.{pdf,png}` — the `phase3_` prefix
avoids clobbering baseline figures.

In [9]:
def _make_comparison_fig(results_subset, feat_type, domain):
    """Build a 2-panel comparison figure for one (feature_type, domain).

    Left panel: bar chart of n_sig_features_q05 per language pair.
    Right panel: bar chart of perm_obs_dist (aggregate profile L2 distance) per pair,
                 with significance stars from perm_p_value.
    """
    if not results_subset:
        return None

    pairs_ordered = [f'{l1}-{l2}' for (l1, l2) in PAIRS]
    pair_nice     = {f'{l1}-{l2}': PAIR_LABELS[(l1, l2)] for (l1, l2) in PAIRS}
    pair_col      = {f'{l1}-{l2}': PAIR_COLORS[(l1, l2)] for (l1, l2) in PAIRS}

    # Index results by pair
    by_pair = {r['pair']: r for r in results_subset}

    n_sig_vals   = [by_pair.get(p, {}).get('n_sig_features_q05', 0) for p in pairs_ordered]
    n_feat_total = next((r['n_features'] for r in results_subset), 1)
    obs_dist_vals = [by_pair.get(p, {}).get('perm_obs_dist', 0.0) for p in pairs_ordered]
    perm_p_vals   = [by_pair.get(p, {}).get('perm_p_value', 1.0) for p in pairs_ordered]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        f'Phase 3 — {DOMAIN_LABELS.get(domain, domain)}: {feat_type} topology features\n'
        f'Cross-linguistic comparison (EN / ES / RU, color domain)',
        fontsize=12, fontweight='bold'
    )

    x = np.arange(len(pairs_ordered))
    colors = [pair_col[p] for p in pairs_ordered]
    labels = [pair_nice[p] for p in pairs_ordered]

    # Left: n_sig_features
    bars0 = axes[0].bar(x, n_sig_vals, color=colors, alpha=0.85, width=0.5)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, fontsize=11)
    axes[0].set_ylabel(f'Significant features (q < {FDR_Q})', fontsize=10)
    axes[0].set_title(f'BH-corrected significant features\n(of {n_feat_total} total)', fontsize=10)
    axes[0].grid(axis='y', linestyle='--', alpha=0.4)
    for bar, val in zip(bars0, n_sig_vals):
        axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + n_feat_total * 0.005,
                     str(val), ha='center', va='bottom', fontsize=9)

    # Right: aggregate profile L2 distance (permutation test)
    bars1 = axes[1].bar(x, obs_dist_vals, color=colors, alpha=0.85, width=0.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, fontsize=11)
    axes[1].set_ylabel('Mean-profile L2 distance', fontsize=10)
    axes[1].set_title('Permutation-test aggregate profile distance\n(* = p < 0.05)', fontsize=10)
    axes[1].grid(axis='y', linestyle='--', alpha=0.4)
    for bar, dist_val, p in zip(bars1, obs_dist_vals, perm_p_vals):
        label = f'{dist_val:.3f}'
        if p < 0.05:
            label += ' *'
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + max(obs_dist_vals) * 0.02,
                     label, ha='center', va='bottom', fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    return fig


saved_figs = []

for feat_type, flat_dict in FEAT_TYPE_MAP:
    available_results = [r for r in all_results if r['feature_type'] == feat_type]
    if not available_results:
        print(f'[{feat_type}] No results — figure skipped.')
        continue

    for domain in DOMAINS:
        domain_results = [r for r in available_results if r['domain'] == domain]
        if not domain_results:
            continue

        fig = _make_comparison_fig(domain_results, feat_type, domain)
        if fig is None:
            continue
        stem = f'phase3_{feat_type}_comparison_{domain}'
        save_fig(fig, stem)
        saved_figs.append(stem)
        plt.close(fig)

print(f'\nFigures saved: {saved_figs}')

  Saved phase3_thresholds_comparison_color.pdf  +  phase3_thresholds_comparison_color.png
[ripser] No results — figure skipped.
[templates] No results — figure skipped.

Figures saved: ['phase3_thresholds_comparison_color']


## OOV / coverage caveats

mBERT uses WordPiece tokenization with no explicit OOV — all tokens get at least a
`[UNK]` token. Coverage concerns are different from the MUSE baseline:

- The threshold-feature quality depends on the KWIC corpus having n ≥ 150 samples
  per term. See `bd show ph-project-bcy` for the one documented under-target term
  (Russian *фиолетовый*, n=104).
- Ripser and template features additionally depend on `mbert_attention_ripser.ipynb`
  having been run on GPU. If those files are absent, the notebook gracefully skips
  those branches — see the "Availability summary" cell above.
- No term is fully OOV from mBERT's vocabulary; subword tokenization ensures coverage
  for all Russian, Spanish, and English color terms in the canon list.

In [10]:
# ── Coverage / sample-count summary ─────────────────────────────────────────
print('=== mBERT KWIC sample counts per (lang, domain) ===')
print('(Loaded from flattened feature matrices — reflects what was actually processed)\n')

for domain in DOMAINS:
    print(f'Domain: {domain}')
    for lang in LANGUAGES:
        ft = flat_thresholds.get((lang, domain))
        n = len(ft) if ft is not None else 'N/A (thresholds not found)'
        # Documented under-target term
        note = ''
        if lang == 'ru' and domain == 'color':
            note = '  [note: фиолетовый n=104 < 150 target, kept; see bd show ph-project-bcy]'
        print(f'  {lang}: N_kwic = {n}{note}')
    print()

=== mBERT KWIC sample counts per (lang, domain) ===
(Loaded from flattened feature matrices — reflects what was actually processed)

Domain: color
  en: N_kwic = 2200
  es: N_kwic = 2161
  ru: N_kwic = 2267  [note: фиолетовый n=104 < 150 target, kept; see bd show ph-project-bcy]



## Discussion

The Phase 3 comparison tests whether mBERT's contextual attention topology differs
across English, Spanish, and Russian color vocabulary at the feature level.

The **threshold features** (12 layers × 12 heads × 6 thresholds × 6 stats = 5184 features)
capture the graph-theoretic properties of per-sentence attention matrices at multiple
sparsification thresholds. Features that show significant Mann-Whitney U differences
(after Benjamini-Hochberg FDR correction) indicate that the two languages produce
systematically different attention structures at some combination of layer, head, and
threshold for at least some sentences.

The **permutation test** on aggregate L2 profile distance checks whether the observed
between-language difference is larger than would be expected by chance (under label
permutation). A small p-value here means the overall attentional topology differs
in aggregate, not just in isolated features.

**Scope boundary with bt5 (Phase 4):** This notebook stops at the basic distributional
comparison (Mann-Whitney U + permutation). Barcode-distance metrics (bottleneck,
Wasserstein) and the four pre-registered prediction tests (P1–P4) are owned by
`ph-project-bt5` and will be added once this v1 comparison is reviewed.

In [11]:
# ── Final sanity check: verify all expected outputs exist ───────────────────
import os

# Required outputs (always expected if thresholds tensors were available)
expected = []

if _thold_available:
    expected.append(COMP_DIR / 'summary.csv')
    for domain in DOMAINS:
        for ext in ['pdf', 'png']:
            expected.append(FIGURES_DIR / f'phase3_thresholds_comparison_{domain}.{ext}')

# Ripser / template figures only expected if those tensors were available
if _ripser_available:
    for domain in DOMAINS:
        for ext in ['pdf', 'png']:
            expected.append(FIGURES_DIR / f'phase3_ripser_comparison_{domain}.{ext}')

if _template_available:
    for domain in DOMAINS:
        for ext in ['pdf', 'png']:
            expected.append(FIGURES_DIR / f'phase3_templates_comparison_{domain}.{ext}')

missing = [str(f) for f in expected if not f.exists()]

if missing:
    print('MISSING FILES:')
    for m in missing:
        print(f'  {m}')
    raise FileNotFoundError(f'{len(missing)} expected output file(s) missing.')
elif not expected:
    print('No feature tensors were available — nothing to verify.')
    print('Re-run mbert_attention_thresholds.ipynb first, then re-execute this notebook.')
else:
    print(f'All {len(expected)} expected output file(s) present.')
    print()

    if (COMP_DIR / 'summary.csv').exists():
        df_check = pd.read_csv(COMP_DIR / 'summary.csv')
        print(f'summary.csv: {len(df_check)} rows, {len(df_check.columns)} columns')
        print(df_check.to_string(index=False))
        print()

    print('Output figures:')
    for f in sorted(FIGURES_DIR.glob('phase3_*.pdf')):
        size_kb = f.stat().st_size // 1024
        print(f'  {f.name}  ({size_kb} kB)')

All 3 expected output file(s) present.

summary.csv: 3 rows, 13 columns
feature_type  pair domain lang_a lang_b  n_features  n_kwic_a  n_kwic_b  n_sig_features_q05  frac_sig_features  perm_obs_dist  perm_p_value           top5_feature_idx
  thresholds en-es  color     en     es        5184      2200      2161                3914             0.7550     582.138924         0.001  [300, 153, 156, 438, 439]
  thresholds en-ru  color     en     ru        5184      2200      2267                4210             0.8121     962.332984         0.001  [372, 289, 296, 297, 366]
  thresholds ru-es  color     ru     es        5184      2267      2161                4050             0.7812     705.887967         0.001 [392, 1199, 301, 374, 372]

Output figures:
  phase3_thresholds_comparison_color.pdf  (28 kB)
